# Transformer Resource Accounting Analysis

This notebook analyzes the computational cost (FLOPs) and memory requirements of GPT-2 models.

## Key Concept: FLOP Counting Rule
For matrix multiplication **A ∈ ℝ^(m×n)** and **B ∈ ℝ^(n×p)**:
- **AB requires 2mnp FLOPs**
- Why? Each element (AB)[i,j] = A[i,:] · B[:,j] needs n multiplications + n additions = 2n ops
- Total elements in AB: m×p
- Total FLOPs: 2n × (m×p) = 2mnp

## Transformer Architecture Overview
From our codebase, a Transformer LM consists of:
1. **Token embeddings** (lookup, no matrix multiply)
2. **Position embeddings** (lookup, no matrix multiply)  
3. **N Transformer blocks**, each with:
   - RMSNorm (element-wise, no matrix multiply)
   - **Multi-head self-attention** (4 matrix multiplies)
   - RMSNorm (element-wise, no matrix multiply)
   - **Position-wise feedforward** (2 matrix multiplies)
4. **Final RMSNorm** (element-wise, no matrix multiply)
5. **Language model head** (1 matrix multiply)


In [1]:
import torch
import math
from dataclasses import dataclass
from typing import Dict, Tuple

@dataclass
class GPTConfig:
    """Configuration for GPT models"""
    vocab_size: int
    context_length: int  
    num_layers: int
    d_model: int
    num_heads: int
    d_ff: int
    
    def __post_init__(self):
        assert self.d_model % self.num_heads == 0, "d_model must be divisible by num_heads"
        self.d_head = self.d_model // self.num_heads

# GPT-2 model configurations
configs = {
    "gpt2_small": GPTConfig(50257, 1024, 12, 768, 12, 3072),
    "gpt2_medium": GPTConfig(50257, 1024, 24, 1024, 16, 4096), 
    "gpt2_large": GPTConfig(50257, 1024, 36, 1280, 20, 5120),
    "gpt2_xl": GPTConfig(50257, 1024, 48, 1600, 25, 6400)
}

print("Model configurations loaded:")
for name, config in configs.items():
    print(f"{name}: {config.num_layers}L, {config.d_model}D, {config.num_heads}H, {config.d_ff}FF")


Model configurations loaded:
gpt2_small: 12L, 768D, 12H, 3072FF
gpt2_medium: 24L, 1024D, 16H, 4096FF
gpt2_large: 36L, 1280D, 20H, 5120FF
gpt2_xl: 48L, 1600D, 25H, 6400FF


## Step 1: Identify Matrix Multiplications

Let's systematically identify all matrix multiplies in our Transformer LM. Based on our codebase analysis:

### Per Transformer Block (occurs N times):

#### Multi-Head Self-Attention:
1. **Q projection**: `in_features @ q_weights.T`
   - Input: `(batch, seq_len, d_model)` 
   - Weight: `(d_model, d_model)`  [Note: consolidates all heads]
   - Output: `(batch, seq_len, d_model)`

2. **K projection**: `in_features @ k_weights.T`
   - Same dimensions as Q projection

3. **V projection**: `in_features @ v_weights.T` 
   - Same dimensions as Q projection

4. **Attention computation**: `QK^T` in `scaled_dot_product_attention`
   - Q,K: `(batch, num_heads, seq_len, d_head)`
   - QK^T: `(batch, num_heads, seq_len, seq_len)`

5. **Attention output**: `Attention_weights @ V`
   - Attention: `(batch, num_heads, seq_len, seq_len)`
   - V: `(batch, num_heads, seq_len, d_head)`
   - Output: `(batch, num_heads, seq_len, d_head)`

6. **Output projection**: `concat_heads @ output_proj.T`
   - Input: `(batch, seq_len, d_model)`
   - Weight: `(d_model, d_model)`
   - Output: `(batch, seq_len, d_model)`

#### Position-wise Feedforward:
7. **First FFN layer (w1)**: `input @ w1.T`
   - Input: `(batch, seq_len, d_model)`
   - Weight: `(d_ff, d_model)`  
   - Output: `(batch, seq_len, d_ff)`

8. **Second FFN layer (w2)**: `hidden @ w2.T`
   - Input: `(batch, seq_len, d_ff)`
   - Weight: `(d_model, d_ff)`
   - Output: `(batch, seq_len, d_model)`

### Final Layer:
9. **Language model head**: `final_features @ lm_head.T`
   - Input: `(batch, seq_len, d_model)`
   - Weight: `(vocab_size, d_model)`
   - Output: `(batch, seq_len, vocab_size)`

**Total per forward pass: 8 × N + 1 matrix multiplies** (where N = num_layers)


## Problem (a): GPT-2 XL Parameter Count & Memory

**Your task**: Calculate the number of trainable parameters and memory requirements.

### Parameter Counting Framework:

For each weight matrix, parameters = rows × columns.

**Per Transformer Block:**
- Q projection: `(d_model, d_model)` 
- K projection: `(d_model, d_model)`
- V projection: `(d_model, d_model)`
- Output projection: `(d_model, d_model)`
- FFN W1: `(d_ff, d_model)`
- FFN W2: `(d_model, d_ff)`  
- RMSNorm weights: `d_model` + `d_model` = `2 × d_model`

**Additional Parameters:**
- Token embeddings: `(vocab_size, d_model)`
- Position embeddings: `(context_length, d_model)`
- Final RMSNorm: `d_model`
- LM head: `(vocab_size, d_model)`

**Memory**: Each parameter is a 32-bit float = 4 bytes


In [3]:
def calculate_parameters(config: GPTConfig) -> Dict[str, int]:
    """Calculate trainable parameters for a GPT model.
    
    TODO: Fill in the parameter calculations for each component.
    Hint: For each weight matrix, parameters = rows × columns
    """
    params = {}
    
    # TODO: Calculate parameters per transformer block
    # Attention weights (Q, K, V, output projection)
    attention_params_per_block = 4 * config.d_model * config.d_model  # Fill this in
    
    # FFN weights (w1, w2) 
    ffn_params_per_block = config.d_model * config.d_ff + config.d_ff * config.d_model  # Fill this in
    
    # RMSNorm weights (2 per block: ln1, ln2)
    norm_params_per_block = 2 * config.d_model  # Fill this in
    
    # Total per block
    params_per_block = attention_params_per_block + ffn_params_per_block + norm_params_per_block
    params['transformer_blocks'] = params_per_block * config.num_layers
    
    # TODO: Calculate embedding and output parameters
    params['token_embeddings'] = config.vocab_size * config.d_model  # Fill this in
    params['position_embeddings'] = config.context_length * config.d_model  # Fill this in  
    params['final_norm'] = config.d_model  # Fill this in
    params['lm_head'] = config.vocab_size * config.d_model  # Fill this in
    
    params['total'] = sum(params.values())
    return params

# Test with GPT-2 XL
gpt2_xl = configs['gpt2_xl']
xl_params = calculate_parameters(gpt2_xl)

print("GPT-2 XL Parameters:")
for component, count in xl_params.items():
    if component != 'total':
        print(f"  {component}: {count:,}")
print(f"Total: {xl_params['total']:,}")

# Memory calculation
memory_bytes = xl_params['total'] * 4  # 4 bytes per float32
memory_gb = memory_bytes / (1024**3)
print(f"\nMemory requirement: {memory_gb:.2f} GB")


GPT-2 XL Parameters:
  transformer_blocks: 1,474,713,600
  token_embeddings: 80,411,200
  position_embeddings: 1,638,400
  final_norm: 1,600
  lm_head: 80,411,200
Total: 1,637,176,000

Memory requirement: 6.10 GB


## FLOP Calculation Framework

Now let's calculate FLOPs for each matrix multiply. Remember: **A ∈ ℝ^(m×n)** × **B ∈ ℝ^(n×p)** requires **2mnp FLOPs**.

### Key Dimensions (for batch_size=1, seq_len=context_length):
- Input sequence: `(1, seq_len, d_model)`
- After attention: `(1, seq_len, d_model)` 
- After FFN: `(1, seq_len, d_model)`

### FLOP Analysis Template:

**Per Transformer Block:**

1. **Q projection**: `(1×seq_len×d_model) @ (d_model×d_model)ᵀ`
   - Effective: `(1×seq_len×d_model) @ (d_model×d_model)`
   - FLOPs = `2 × 1 × seq_len × d_model × d_model`

2. **K, V projections**: Same as Q projection

3. **Attention QKᵀ**: For each head, `(seq_len×d_head) @ (d_head×seq_len)`
   - Total across heads: `num_heads × 2 × seq_len × d_head × seq_len`
   - Since `d_head = d_model/num_heads`: `2 × seq_len² × d_model`

4. **Attention output**: For each head, `(seq_len×seq_len) @ (seq_len×d_head)`  
   - Total: `2 × num_heads × seq_len × seq_len × d_head = 2 × seq_len² × d_model`

5. **Output projection**: Same as Q projection

6. **FFN W1**: `(1×seq_len×d_model) @ (d_model×d_ff)ᵀ = 2 × seq_len × d_model × d_ff`

7. **FFN W2**: `(1×seq_len×d_ff) @ (d_ff×d_model)ᵀ = 2 × seq_len × d_ff × d_model`

**Final LM Head**: `(1×seq_len×d_model) @ (d_model×vocab_size)ᵀ = 2 × seq_len × d_model × vocab_size`


In [ ]:
def calculate_flops(config: GPTConfig, batch_size: int = 1) -> Dict[str, int]:
    """Calculate FLOPs for a forward pass through the Transformer LM."""
    seq_len = config.context_length  # Full context length
    flops = {}
    
    # Per transformer block calculations
    # Q, K, V projections - each is (batch×seq_len×d_model) @ (d_model×d_model)
    qkv_flops_per_block = 3 * 2 * batch_size * seq_len * config.d_model * config.d_model
    
    # Attention QK^T computation - results in seq_len² scaling
    attention_qk_flops_per_block = 2 * batch_size * seq_len * seq_len * config.d_model
    
    # Attention output computation - also seq_len² scaling
    attention_output_flops_per_block = 2 * batch_size * seq_len * seq_len * config.d_model
    
    # Output projection - (batch×seq_len×d_model) @ (d_model×d_model)
    output_proj_flops_per_block = 2 * batch_size * seq_len * config.d_model * config.d_model
    
    # FFN W1 - (batch×seq_len×d_model) @ (d_model×d_ff)
    ffn_w1_flops_per_block = 2 * batch_size * seq_len * config.d_model * config.d_ff
    
    # FFN W2 - (batch×seq_len×d_ff) @ (d_ff×d_model)
    ffn_w2_flops_per_block = 2 * batch_size * seq_len * config.d_ff * config.d_model
    
    # Group components
    flops['attention_projections'] = (qkv_flops_per_block + output_proj_flops_per_block) * config.num_layers
    flops['attention_computation'] = (attention_qk_flops_per_block + attention_output_flops_per_block) * config.num_layers
    flops['feedforward'] = (ffn_w1_flops_per_block + ffn_w2_flops_per_block) * config.num_layers
    
    # LM head - (batch×seq_len×d_model) @ (d_model×vocab_size)
    flops['lm_head'] = 2 * batch_size * seq_len * config.d_model * config.vocab_size
    
    flops['total'] = sum(flops.values())
    return flops


# Test with GPT-2 XL  
xl_flops = calculate_flops(gpt2_xl)

print("GPT-2 XL FLOPs per forward pass:")
for component, count in xl_flops.items():
    if component != 'total':
        proportion = count / xl_flops['total'] * 100 if xl_flops['total'] > 0 else 0
        print(f"  {component}: {count:,} ({proportion:.1f}%)")
print(f"Total: {xl_flops['total']:,}")


GPT-2 XL FLOPs per forward pass:
  attention_projections: 1,006,632,960,000 (28.7%)
  attention_computation: 322,122,547,200 (9.2%)
  feedforward: 2,013,265,920,000 (57.4%)
  lm_head: 164,682,137,600 (4.7%)
Total: 3,506,703,564,800


## Problems (b), (c), (d): Comparative Analysis

Once you've filled in the calculations above, use the functions below to analyze:

1. **Problem (b)**: Which components require the most FLOPs in GPT-2 XL?
2. **Problem (c)**: How do FLOP proportions change across model sizes?  
3. **Problem (d)**: How does increasing context length affect FLOPs?

### Key Insights to Discover:

**Attention vs. FFN**: 
- Attention: O(seq_len²) scaling due to pairwise token interactions
- FFN: O(seq_len) scaling, linear in sequence length
- **Question**: At what sequence length does attention dominate?

**Parameter scaling**: 
- d_model appears in O(d_model²) terms (projections)
- d_ff often ≈ 4×d_model, so FFN has O(d_model × d_ff) ≈ O(4×d_model²)
- **Question**: How does this affect the balance as models grow?

**Context length scaling**:
- Most operations scale as O(seq_len) 
- Attention computation scales as O(seq_len²)
- **Question**: What happens when we go from 1K to 16K context?


In [5]:
# Comparative analysis functions

def analyze_all_models():
    """Analyze FLOP proportions across all GPT-2 model sizes."""
    print("FLOP Analysis Across GPT-2 Model Sizes:")
    print("=" * 60)
    
    for name, config in configs.items():
        flops = calculate_flops(config)
        if flops['total'] > 0:  # Only if you've filled in the calculations
            print(f"\n{name.upper()}:")
            for component, count in flops.items():
                if component != 'total':
                    proportion = count / flops['total'] * 100
                    print(f"  {component}: {proportion:.1f}%")

def analyze_context_length_scaling():
    """Analyze how increasing context length affects FLOPs."""
    print("\nContext Length Scaling Analysis (GPT-2 XL):")
    print("=" * 50)
    
    context_lengths = [1024, 2048, 4096, 8192, 16384]
    
    for context_len in context_lengths:
        # Create modified config with different context length
        modified_config = GPTConfig(
            vocab_size=gpt2_xl.vocab_size,
            context_length=context_len,
            num_layers=gpt2_xl.num_layers,
            d_model=gpt2_xl.d_model,
            num_heads=gpt2_xl.num_heads,
            d_ff=gpt2_xl.d_ff
        )
        
        flops = calculate_flops(modified_config)
        if flops['total'] > 0:
            print(f"\nContext Length {context_len}:")
            for component, count in flops.items():
                if component != 'total':
                    proportion = count / flops['total'] * 100
                    print(f"  {component}: {proportion:.1f}%")

# Run the analyses (will show results once you fill in the FLOP calculations)
analyze_all_models()
analyze_context_length_scaling()


FLOP Analysis Across GPT-2 Model Sizes:

GPT2_SMALL:
  attention_projections: 12.0%
  attention_computation: 47.9%
  feedforward: 23.9%
  lm_head: 16.3%

GPT2_MEDIUM:
  attention_projections: 13.3%
  attention_computation: 53.3%
  feedforward: 26.6%
  lm_head: 6.8%

GPT2_LARGE:
  attention_projections: 13.7%
  attention_computation: 55.0%
  feedforward: 27.5%
  lm_head: 3.7%

GPT2_XL:
  attention_projections: 14.0%
  attention_computation: 55.9%
  feedforward: 27.9%
  lm_head: 2.3%

Context Length Scaling Analysis (GPT-2 XL):

Context Length 1024:
  attention_projections: 14.0%
  attention_computation: 55.9%
  feedforward: 27.9%
  lm_head: 2.3%

Context Length 2048:
  attention_projections: 9.0%
  attention_computation: 71.7%
  feedforward: 17.9%
  lm_head: 1.5%

Context Length 4096:
  attention_projections: 5.2%
  attention_computation: 83.5%
  feedforward: 10.4%
  lm_head: 0.9%

Context Length 8192:
  attention_projections: 2.8%
  attention_computation: 91.0%
  feedforward: 5.7%
  lm

## Educational Insights & Next Steps

### What You Should Discover:

1. **FFN Dominance**: In most configurations, feedforward layers consume ~67% of FLOPs
   - Why? FFN has 2 large matrix multiplies: d_model→d_ff→d_model where d_ff ≈ 4×d_model

2. **Attention Scaling**: Attention's share increases quadratically with context length
   - At short sequences: FFN dominates
   - At very long sequences: Attention can become significant

3. **Model Size Effects**: As models grow larger (more parameters), proportions stay roughly constant
   - This suggests the architectural balance is well-designed

### Deep Learning Concepts Reinforced:

**Computational Complexity**:
- Linear operations: O(d_model²) scaling
- Attention: O(seq_len²) bottleneck for long sequences
- Understanding these trade-offs guides architectural choices

**Memory vs. Compute Trade-offs**:
- Parameters determine memory (storage)
- FLOPs determine compute time 
- Different optimizations target different bottlenecks

**Architectural Design Principles**:
- FFN width (d_ff) vs. number of layers
- Attention heads vs. head dimension
- Context length vs. computational cost

### Follow-up Questions to Explore:

1. How do these calculations change with different precision (float16 vs. float32)?
2. What about gradient computation during training (roughly 2x forward pass FLOPs)?
3. How do modern optimizations (FlashAttention, tensor parallelism) affect these numbers?
4. What's the relationship between FLOPs and actual wall-clock time on GPUs?

**Congratulations!** You've completed a foundational analysis of Transformer computational costs. This understanding is crucial for scaling language models efficiently.
